# Performer
You can __[download](https://arxiv.org/pdf/2009.14794)__ and read the paper. Also, there is a __[video](https://www.youtube.com/watch?v=xJrKIPwVwGM&t=76s)__ describes paper.

Here we want to implement Performer on __[LLAMA3](https://huggingface.co/meta-llama/Llama-3.2-1B)__ and compare it with Vanilla Transformer Attention mechanism.
## What expect?
Vanilla attention is O(L^2) and performer is O(L). There for, Performer should be faster and need less time and memmory.
## A brief look at the formulas
Vanilla transformer mechanism uses formula below to calculate:
$$\text{Attention}(Q, K, V) = \text{softmax} \left( \frac{QK^T}{\sqrt{d_k}} \right) V$$
For Performer attention we use (FAVOR+):
$$\text{Attention}_{\text{FAVOR}+}(Q, K, V) = \frac{\Phi(Q) \left( \Phi(K)^\top V \right)}{\Phi(Q) \left( \Phi(K)^\top \mathbf{1} \right)}$$
## Implement Libraries that you need
Here we implemented Performer attention.

NOTE: Even we implemented vanilla attention, you can load LLAMA3 with vanilla attention. Just use line below:
```python
attn_implementation="eager"
```
### You need to run Language model downloaded from hugging face?
If You need some language model from hugging face, you need to implement huggingface_hub and use login function.

Use code below to implement it:
```python
from huggingface_hub import login
login()
```

You need __[Access Token](https://huggingface.co/docs/hub/en/security-tokens)__ for that. You can __[crete your access token](https://huggingface.co/settings/tokens)__ by your own.

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
import math
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb
from huggingface_hub import login

login()

## Load your model
You can load any model that you want. Just use huggingface link pass it as funtion input. As default model, we set it LLAMAA-3.2-1B and quantized it to 4bit.
### Want to use small models?
Take it easy. Just disable quantization using quantize parameter. Make it false
### LITTE TIP
If you use small models, you may not get your prefered output. Use models that can handel inputs with more than 4096 inputs (The reason is mentioned in next cells).

In [ ]:
def generate_model(model_id="meta-llama/Llama-3.2-1B", quantize=True, device="cuda"):

    if quantize:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        bnb_config = None
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    ).to(device)
    model = prepare_model_for_kbit_training(model)
    return model, tokenizer

model, tokenizer = generate_model()